In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/20 19:04:29 WARN Utils: Your hostname, Chirags-Laptop.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.117 instead (on interface en0)
26/06/20 19:04:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 19:04:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
SparkSession.getActiveSession()

In [9]:
!curl -O https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [4]:
!wc -l fhvhv_tripdata_2021-01.csv

 11908469 fhvhv_tripdata_2021-01.csv


In [13]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [15]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [17]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [19]:
# We are creating a small sub set of data to use pandas from so that it can manage the amount of data
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [23]:
import pandas as pd

In [25]:
df_pandas = pd.read_csv('head.csv')

In [27]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [29]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [5]:
from pyspark.sql import types

In [6]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [7]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [8]:
# Now we have the datatypes corrected 
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [9]:
# We'll partition our data because each partition would go to spark cluster to process. 
# if we just take the input as it is then number of partition would be equal to the number of file present when creating "df" i.e 1
# To parallize this work we do this 

df = df.repartition(24)

In [10]:
# This will write 24 parquet file in the given directory
df.write.parquet('fhvhv/2021/01/')

26/06/20 19:14:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/20 19:14:40 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/20 19:14:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [73]:
spark.stop()

In [7]:
df = spark.read.parquet('fhvhv/2021/01/')

In [11]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [15]:
df.select('dispatching_base_num','pickup_datetime',"dropoff_datetime") \
.filter(df.hvfhs_license_num == 'HV0003') \
.show()

+--------------------+-------------------+-------------------+
|dispatching_base_num|    pickup_datetime|   dropoff_datetime|
+--------------------+-------------------+-------------------+
|              B02835|2021-01-01 04:54:48|2021-01-01 05:08:21|
|              B02764|2021-01-03 11:10:58|2021-01-03 11:18:51|
|              B02883|2021-01-03 20:18:04|2021-01-03 20:28:03|
|              B02836|2021-01-03 18:49:50|2021-01-03 18:59:51|
|              B02878|2021-01-04 10:12:06|2021-01-04 10:26:08|
|              B02865|2021-01-02 00:21:39|2021-01-02 00:32:52|
|              B02764|2021-01-03 14:59:05|2021-01-03 15:05:30|
|              B02867|2021-01-04 21:42:33|2021-01-04 21:48:31|
|              B02872|2021-01-03 17:02:18|2021-01-03 17:08:34|
|              B02871|2021-01-02 16:26:53|2021-01-02 16:29:28|
|              B02876|2021-01-04 18:42:29|2021-01-04 18:50:40|
|              B02889|2021-01-02 13:06:26|2021-01-02 13:15:08|
|              B02880|2021-01-02 09:09:04|2021-01-02 09

In [21]:
from pyspark.sql import functions as F

In [17]:
# This is a user defined function where if your buisness logic is complex to represent through SQL then we use UDF

def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [19]:
crazy_stuff('B02884')

's/b44'

In [27]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [29]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  s/b13| 2021-01-01|  2021-01-01|          59|          51|
|  e/acc| 2021-01-03|  2021-01-03|          49|          25|
|  a/b43| 2021-01-03|  2021-01-03|         167|         168|
|  e/b14| 2021-01-03|  2021-01-03|          22|          29|
|  e/9ce| 2021-01-03|  2021-01-03|          65|         189|
|  e/b3e| 2021-01-04|  2021-01-04|          97|          13|
|  a/b31| 2021-01-02|  2021-01-02|         254|          51|
|  e/9ce| 2021-01-05|  2021-01-05|          48|         170|
|  e/9ce| 2021-01-02|  2021-01-02|         254|           3|
|  e/acc| 2021-01-03|  2021-01-03|          82|          82|
|  e/b33| 2021-01-04|  2021-01-04|         137|         233|
|  e/b38| 2021-01-03|  2021-01-03|          40|          33|
|  a/b37| 2021-01-02|  2021-01-02|         126|         126|
|  e/9ce| 2021-01-02|  2